[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Type Affinity


## What you will be able to do

Say what SQLite stores for any value in any column, from the column's declared type and the value's
own type, and check it with `typeof`. Recognize the declared types that quietly change what a column
keeps, and the mixed columns where `WHERE`, `ORDER BY` and `MAX` stop meaning what you expect. Store
bytes in a `BLOB` column, and create `STRICT` tables that refuse a value of the wrong type instead of
keeping it.


## The idea

### The problem

In the **Why sqlite3** notebook, 24 empty readings from a CSV file went into a column declared
`REAL`, and nothing refused them. They were kept as text, and `MAX` then rated an empty string warmer
than every temperature Svalbard recorded in March. Most databases would have refused the insert:
PostgreSQL rejects the text `'hello'` for an integer column with an error. SQLite, by default, takes
the value and keeps it, in whatever form it can.

That is a design, not an accident. A column's declared type is a preference, and the preference does
useful work: it turned every other reading in that file, which arrived as text, into a number. But a
value it cannot turn into a number, such as `n/a`, or `-7,4` written with the decimal comma that
Norwegian software uses, goes in as text beside the numbers. From then on `WHERE celsius < 0` never
counts it, `ORDER BY` puts it after every number, and Python receives a `str` from a column declared
`REAL`. And a declared type does not always mean what it says: a column declared `STRING` stores
Oslo's postal code, `0150`, as the number 150.

### What type affinity is

> Every value SQLite stores has one of five **storage classes**: `NULL`, `INTEGER`, `REAL`, `TEXT` or
> `BLOB`, which the SQL function `typeof` reports. The storage class belongs to the value, not to its
> column, so one column can hold all five. A column's **type affinity** is the storage class it
> prefers, worked out from the name of the type it was declared with. When a value arrives, the
> affinity converts it if nothing is lost, as the text `'4.2'` becomes the number 4.2 in a column
> declared `REAL`, and otherwise stores the value as it came. A **`STRICT`** table makes the
> preference a rule: a value that cannot be converted without loss raises `sqlite3.IntegrityError`.

### Why it works that way

- **The type belongs to the value.** SQLite's documentation calls this flexible typing, and says it
  is a feature of SQLite, not a bug. Values of different kinds can share a column, and a declared
  type only steers what arrives.
- **Affinity is read from the declared type's name, by five rules in order.** A name containing
  `INT` gives INTEGER affinity, `CHAR`, `CLOB` or `TEXT` give TEXT, `BLOB` or no type at all gives
  BLOB, `REAL`, `FLOA` or `DOUB` give REAL, and any other name gives NUMERIC. So `VARCHAR(20)` is
  TEXT, `DATETIME` and `STRING` are NUMERIC, and `FLOATING POINT` is INTEGER, for the `INT` in
  `POINT`.
- **A conversion happens only when nothing is lost.** The numeric affinities turn `'42'` into a
  number and TEXT turns 42 into `'42'`, while `'n/a'`, `''` and `'-7,4'` look like no number, so
  every affinity keeps them as text. BLOB affinity converts nothing at all.
- **The storage classes sort in a fixed order.** `NULL` comes first, then numbers by value, then
  text, then blobs. Text in a column of numbers sorts after all of them, `MAX` returns it, and
  `< 0` is never true for it.
- **Python receives the storage class, not the declaration.** INTEGER arrives as `int`, REAL as
  `float`, TEXT as `str`, BLOB as `bytes` and NULL as `None`, whatever type the column was declared
  with.
- **`STRICT` arrived in SQLite 3.37.0, in 2021.** A STRICT table allows only the type names `INT`,
  `INTEGER`, `REAL`, `TEXT`, `BLOB` and `ANY`, still converts what converts without loss, and
  refuses the rest. A file holding a STRICT table can be opened only by SQLite 3.37.0 or later.

### Where you will meet this

Almost every other database checks types as values arrive. PostgreSQL, in the **asyncpg and
psycopg3, Deep Dive** guide, refuses `'hello'` for an integer column with `invalid input syntax for
type integer`, and the **DuckDB, Deep Dive** guide's database refuses it as well, so SQL that SQLite
accepts can start failing the day it moves. The **Column Types** notebook in the **SQLAlchemy, Deep
Dive** guide shows an ORM converting values in Python before SQLite sees them. Text reaches a
database from everywhere: the **CSV** notebook in the **Files, Paths and Formats** guide showed that
every field `csv` reads is a string, and JSON from an API can carry numbers inside quotes.

### What this notebook covers

- Storage classes, and `typeof`
- Affinity, read from a declared type's name
- How values of different storage classes sort and compare
- What Python receives from each storage class
- `STRICT` tables, and `ANY`
- Bytes in a `BLOB` column, and `blobopen` for reading part of one
- When to use a `STRICT` table, a `CHECK` on `typeof`, or neither
- A loader that turns CSV text into typed values before a `STRICT` table sees them
- Six errors: a station's name in a `STRICT` integer column, a type name `STRICT` does not know,
  Python comparing a `str` with a `float`, readings with no declared type, a postal code in a column
  declared `STRING`, and `CAST` reading only the start of the text

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE flexible (hours INTEGER)")
conn.execute("CREATE TABLE typed (hours INTEGER) STRICT")

conn.executemany("INSERT INTO flexible VALUES (?)", [("24",), ("hello",)])
print(conn.execute("SELECT hours, typeof(hours) FROM flexible").fetchall())

conn.execute("INSERT INTO typed VALUES (?)", ("24",))
try:
    conn.execute("INSERT INTO typed VALUES (?)", ("hello",))
except sqlite3.IntegrityError as error:
    print("IntegrityError:", error)
print(conn.execute("SELECT hours, typeof(hours) FROM typed").fetchall())
conn.close()
```

```
[(24, 'integer'), ('hello', 'text')]
IntegrityError: cannot store TEXT value in INTEGER column typed.hours
[(24, 'integer')]
```

Two tables, each with a column declared `INTEGER`, given the same two pieces of text. The flexible
table turned `'24'` into the integer 24 and kept `'hello'`, which it could not convert, as text in
its `INTEGER` column. The `STRICT` table converted `'24'` the same way and refused `'hello'` with an
error, so it holds only the integer.


## Setup

Eight imports, and the stations' year, built into the two tables the **Tables and Queries** notebook
designed.

- `sqlite3` builds the database and runs every statement
- `zlib` compresses a day of readings into bytes for a `BLOB` column
- `csv` and `io` read an export held in a string, for the loader that closes the worked examples
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end


In [1]:
import csv
import io
import math
import shutil
import sqlite3
import zlib
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### Storage classes, and typeof

A column declared with no type converts nothing, which makes it the place to see storage classes on
their own. Five Python values of five different types go into one column, and `typeof` reports the
storage class of each:


In [2]:
conn = sqlite3.connect(DATABASE)
conn.execute("CREATE TABLE anything (value)")
conn.executemany("INSERT INTO anything VALUES (?)", [(-3.5,), (24,), ("-3.5",), (b"\x78\x9c",), (None,)])

for value, storage in conn.execute("SELECT value, typeof(value) FROM anything"):
    print(f"{value!r:<10} {storage}")


-3.5       real
24         integer
'-3.5'     text
b'x\x9c'   blob
None       null


One column holds all five, and the text `'-3.5'` sits beside the number -3.5 as a different value.
That is SQLite's model: a storage class belongs to a value, and a column is a place to put values.

### Affinity, read from the declared type

A declared type gives a column its affinity, and SQLite reads the affinity from the type's name by
five rules, checked in order:

| If the declared type's name | the affinity is | as for |
|---|---|---|
| contains `INT` | INTEGER | `INTEGER`, `INT`, `BIGINT`, and `FLOATING POINT` |
| contains `CHAR`, `CLOB` or `TEXT` | TEXT | `TEXT`, `VARCHAR(20)` |
| contains `BLOB`, or there is no type | BLOB | `BLOB`, and a column declared with no type |
| contains `REAL`, `FLOA` or `DOUB` | REAL | `REAL`, `FLOAT`, `DOUBLE` |
| matches none of those | NUMERIC | `NUMERIC`, `DECIMAL(5,2)`, `BOOLEAN`, `DATETIME`, `STRING` |

The same four values go into a column declared eight different ways, and `typeof` shows what every
column kept:


In [3]:
DECLARED = ["INTEGER", "REAL", "TEXT", "", "NUMERIC", "VARCHAR(20)", "STRING", "FLOATING POINT"]
VALUES = ["42", "4.5", 42, "n/a"]

for declared in DECLARED:
    conn.execute("DROP TABLE IF EXISTS sample")
    conn.execute(f"CREATE TABLE sample (value {declared})")
    conn.executemany("INSERT INTO sample VALUES (?)", [(value,) for value in VALUES])
    kept = [f"{value!r} {storage}" for value, storage in conn.execute("SELECT value, typeof(value) FROM sample")]
    print(f"{declared or '(no type)':<15}", " | ".join(kept))


INTEGER         42 integer | 4.5 real | 42 integer | 'n/a' text
REAL            42.0 real | 4.5 real | 42.0 real | 'n/a' text
TEXT            '42' text | '4.5' text | '42' text | 'n/a' text
(no type)       '42' text | '4.5' text | 42 integer | 'n/a' text
NUMERIC         42 integer | 4.5 real | 42 integer | 'n/a' text
VARCHAR(20)     '42' text | '4.5' text | '42' text | 'n/a' text
STRING          42 integer | 4.5 real | 42 integer | 'n/a' text
FLOATING POINT  42 integer | 4.5 real | 42 integer | 'n/a' text


Every numeric affinity turned the text `'42'` into a number, TEXT turned the number 42 into text,
and the column with no type kept every value as it came. Nothing could make a number of `'n/a'`, so
every column kept it as text, `INTEGER` and `REAL` included. `STRING` behaved like `NUMERIC`, since
no rule matches its name, and `FLOATING POINT` like `INTEGER`. The names in `DECLARED` are written
into the SQL from a fixed list in the program, as the **Parameters** notebook allows for names.

### How the storage classes sort and compare

`NULL` sorts first, numbers next by value, then text in text order, then blobs, and a comparison
converts a value only when one side has a numeric affinity. The same six values go into a column
declared `REAL` and a column with no type:


In [4]:
MIXED = [(-3.5,), ("-4.1",), ("n/a",), (None,), ("10",), (9.0,)]

for table, declared in [("real_column", "REAL"), ("untyped_column", "")]:
    conn.execute(f"CREATE TABLE {table} (celsius {declared})")
    conn.executemany(f"INSERT INTO {table} VALUES (?)", MIXED)
    ordered = [celsius for (celsius,) in conn.execute(f"SELECT celsius FROM {table} ORDER BY celsius")]
    lowest, highest = conn.execute(f"SELECT MIN(celsius), MAX(celsius) FROM {table}").fetchone()
    equal = conn.execute(f"SELECT COUNT(*) FROM {table} WHERE celsius = -4.1").fetchone()[0]
    print(f"{table:<15} ORDER BY: {ordered}")
    print(f"{'':<15} MIN {lowest!r}, MAX {highest!r}, rows equal to -4.1: {equal}")


real_column     ORDER BY: [None, -4.1, -3.5, 9.0, 10.0, 'n/a']
                MIN -4.1, MAX 'n/a', rows equal to -4.1: 1
untyped_column  ORDER BY: [None, -3.5, 9.0, '-4.1', '10', 'n/a']
                MIN -3.5, MAX 'n/a', rows equal to -4.1: 0


In the `REAL` column, `'-4.1'` and `'10'` became numbers on the way in, so they sorted among the
numbers, and only `'n/a'` stayed text: it sorted last, and `MAX` returned it as the warmest reading,
as the empty strings in the **Why sqlite3** notebook did. The column with no type converted nothing.
`'-4.1'` and `'10'` sorted after every number, in text order, `MIN` missed the coldest reading, and
`= -4.1` found no row, since neither side of that comparison has an affinity to convert the text.

### What Python receives

The storage class decides the Python type, whatever the column was declared as, so a column declared
`REAL` can hand Python a `str`:


In [5]:
received = [(celsius, type(celsius).__name__) for (celsius,) in conn.execute("SELECT celsius FROM real_column")]

print(received)


[(-3.5, 'float'), (-4.1, 'float'), ('n/a', 'str'), (None, 'NoneType'), (10.0, 'float'), (9.0, 'float')]


Every value that arrived as REAL is a `float`, `NULL` is `None`, and `'n/a'` is a `str`. Code that
treats everything from this column as a number fails on that one value, far from where it was stored,
which one of the Common errors shows.

### STRICT tables

`STRICT`, written after the closing parenthesis of `CREATE TABLE`, turns a table's types into rules.
A STRICT table still converts what converts without loss, and raises `sqlite3.IntegrityError` for
anything else. It accepts only six type names, and `NULL` goes into any column that is not
`NOT NULL`:


In [6]:
conn.execute("""
    CREATE TABLE strict_readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL,
        hour       TEXT NOT NULL,
        celsius    REAL
    ) STRICT
""")

for celsius in [-3.5, "-3.5", -3, None, "n/a", b"\x00"]:
    try:
        cursor = conn.execute("INSERT INTO strict_readings (station_id, hour, celsius) VALUES (1, '2025-01-01T00:00', ?)",
                              (celsius,))
        stored = conn.execute("SELECT celsius, typeof(celsius) FROM strict_readings WHERE id = ?",
                              (cursor.lastrowid,)).fetchone()
        print(f"{celsius!r:<8} stored as {stored}")
    except sqlite3.IntegrityError as error:
        print(f"{celsius!r:<8} refused: {error}")


-3.5     stored as (-3.5, 'real')
'-3.5'   stored as (-3.5, 'real')
-3       stored as (-3.0, 'real')
None     stored as (None, 'null')
'n/a'    refused: cannot store TEXT value in REAL column strict_readings.celsius
b'\x00'  refused: cannot store BLOB value in REAL column strict_readings.celsius


The text `'-3.5'` and the integer -3 both became REAL, as a flexible `REAL` column would have made
them, and `None` went in as `NULL`. `'n/a'` and the bytes could not become a number, and where a
flexible table would have kept them, the STRICT table refused them.

`ANY` is the sixth type name, for a column meant to hold anything. In a STRICT table it keeps every
value exactly as given, so the text `'0150'` stays text. `pragma_table_list` reports which tables are
STRICT:


In [7]:
conn.execute("CREATE TABLE codes (town TEXT NOT NULL, code ANY) STRICT")
conn.execute("INSERT INTO codes VALUES (?, ?)", ("Oslo", "0150"))

print(conn.execute("SELECT town, code, typeof(code) FROM codes").fetchone())
print(conn.execute("""
    SELECT name, strict FROM pragma_table_list
    WHERE name IN ('readings', 'strict_readings', 'codes')
    ORDER BY name
""").fetchall())


('Oslo', '0150', 'text')
[('codes', 1), ('readings', 0), ('strict_readings', 1)]


`strict` is 1 for a STRICT table and 0 for a flexible one. A table is STRICT from the `CREATE TABLE`
that makes it: `ALTER TABLE` can rename a table and add, rename or drop a column, but it cannot make
a table STRICT, so an existing table takes the rebuild that the **Changing a Schema** notebook
shows.

### Bytes in a BLOB column

A `BLOB` column holds bytes exactly as given. Python passes `bytes`, `bytearray` or a `memoryview`,
and receives `bytes`. Here Svalbard's readings for 1 March, written out as CSV text and compressed
with `zlib`, go into a BLOB column and come back intact:


In [8]:
day = conn.execute("""
    SELECT r.hour, r.celsius FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.hour LIKE ? ORDER BY r.hour
""", ("Svalbard", "2025-03-01%")).fetchall()
text = "\n".join(f"{hour},{celsius}" for hour, celsius in day).encode()

conn.execute("""
    CREATE TABLE archives (id INTEGER PRIMARY KEY, station TEXT NOT NULL, day TEXT NOT NULL, data BLOB NOT NULL) STRICT
""")
archive_id = conn.execute("INSERT INTO archives (station, day, data) VALUES (?, ?, ?)",
                          ("Svalbard", "2025-03-01", zlib.compress(text))).lastrowid
conn.commit()

data, storage, size = conn.execute("SELECT data, typeof(data), length(data) FROM archives WHERE id = ?",
                                   (archive_id,)).fetchone()
print(f"{len(text)} bytes of text, stored as {size} bytes of {storage}, returned as {type(data).__name__}")
print(zlib.decompress(data).decode().splitlines()[:2])


543 bytes of text, stored as 132 bytes of blob, returned as bytes
['2025-03-01T00:00,-12.8', '2025-03-01T01:00,-13.0']


`SELECT` hands back a whole blob at once, which wastes memory when only part of a large one is
needed. `conn.blobopen` opens the value in one row as a file-like object, which reads, seeks and
writes in place without loading the rest. Its arguments are the table, the column, and the row's
rowid, which for a table with an `INTEGER PRIMARY KEY` is that key, `id` here:


In [9]:
with conn.blobopen("archives", "data", archive_id, readonly=True) as blob:
    header = blob.read(2)
    length = len(blob)

print("the first two bytes:", header, "of", length)


the first two bytes: b'x\x9c' of 132


Two bytes came back, zlib's header, without the other 130 being read. `blobopen` arrived in Python
3.11. It cannot change a blob's length, so a blob that grows is written again with `UPDATE`.

### STRICT, a CHECK on typeof, or neither

A column's types can be enforced in two ways, or left to the program:

| Write | When | Why |
|---|---|---|
| a `STRICT` table | every new table, when every program that opens the file uses SQLite 3.37.0 or later | every column refuses what it cannot convert, with nothing more to write |
| `CHECK (typeof(celsius) IN ('real', 'null'))` | a table that an older SQLite must still open, or a column to guard in a flexible table | a `CHECK` works in any version, one column at a time |
| a flexible table, or an `ANY` column | an existing database, or a column meant to hold values of different kinds | nothing is refused, so the program checks what it stores |

The default for a new table is `STRICT`. A `CHECK` on `typeof` runs after affinity has converted the
value, so `'4.5'` still passes a `REAL` column's check as the number 4.5, and only values that stayed
text fail. Whichever enforces the types, convert text in Python first, where a program decides what a
decimal comma or `n/a` means, and let the table refuse whatever slips past.

### A loader that types its text before STRICT sees it

The pieces of this notebook in one job: the first readings from Kirkenes arrive as an export from
Norwegian software, with semicolons between fields, decimal commas, empty fields and `n/a`.
`parse_celsius` decides what every piece of text means, `None` for a missing reading and a `float`
for a number, and refuses anything else. The rows go into `strict_readings`, emptied first of the
rows the STRICT example left there, and the table would refuse any text that got past the parser, as
the last lines show:


In [10]:
EXPORT = """station;hour;celsius
Kirkenes;2025-12-01T00:00;-7,4
Kirkenes;2025-12-01T01:00;-7,9
Kirkenes;2025-12-01T02:00;
Kirkenes;2025-12-01T03:00;n/a
Kirkenes;2025-12-01T04:00;-8,3
Kirkenes;2025-12-01T05:00;about -8
"""
MISSING = {"", "n/a"}


def parse_celsius(text):
    """A reading as a float, or None when it is missing. A decimal comma is accepted, and other text raises ValueError."""
    text = text.strip()
    if text in MISSING:
        return None
    return float(text.replace(",", "."))


station_ids = dict(conn.execute("SELECT name, id FROM stations"))
conn.execute("DELETE FROM strict_readings")
loaded, rejected = 0, []
for row in csv.DictReader(io.StringIO(EXPORT), delimiter=";"):
    try:
        celsius = parse_celsius(row["celsius"])
    except ValueError:
        rejected.append((row["hour"], row["celsius"]))
        continue
    conn.execute("INSERT INTO strict_readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 (station_ids[row["station"]], row["hour"], celsius))
    loaded += 1
conn.commit()

print("loaded:", loaded, "rejected:", rejected)
print("stored as:", conn.execute("SELECT typeof(celsius), COUNT(*) FROM strict_readings GROUP BY 1 ORDER BY 1").fetchall())
summary = conn.execute("SELECT MIN(celsius), MAX(celsius), ROUND(AVG(celsius), 2) FROM strict_readings").fetchone()
print("coldest, warmest, mean:", summary)

try:
    conn.execute("INSERT INTO strict_readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 (station_ids["Kirkenes"], "2025-12-01T06:00", "-8,6"))
except sqlite3.IntegrityError as error:
    print("text that skipped the parser:", error)


loaded: 5 rejected: [('2025-12-01T05:00', 'about -8')]
stored as: [('null', 2), ('real', 3)]
coldest, warmest, mean: (-8.3, -7.4, -7.87)
text that skipped the parser: cannot store TEXT value in REAL column strict_readings.celsius


Five rows loaded: three readings stored as REAL and two missing ones as `NULL`, which `MIN`, `MAX`
and `AVG` skip. `about -8` was neither a number nor a known way of writing a missing reading, so the
parser set it aside for a person to look at instead of guessing. And the insert that bypassed the
parser, with `-8,6` still as text, was refused by the table.

### Where each part came from

| In the loader | What it relies on | The section that showed it |
|---|---|---|
| `celsius REAL` in a `STRICT` table | a column that refuses text it cannot convert | STRICT tables |
| `parse_celsius` turning `'-7,4'` into -7.4 | text that no affinity reads as a number | Affinity, read from the declared type |
| `None` for `''` and `n/a` | `NULL`, which summaries skip, where text would sort after every number | How the storage classes sort and compare |
| a `float` passed as a parameter | a Python type that arrives as the storage class REAL | What Python receives |
| `typeof(celsius)` in the check afterward | the storage class of every stored value | Storage classes, and typeof |
| converting in Python, with the table as a backstop | the default for enforcing types | STRICT, a CHECK on typeof, or neither |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/06-type-affinity-solutions.ipynb).

**1.** Create a table with one column declared `BIGINT`, insert the text `'7'` and the text
`'seven'`, and print each value with its `typeof`.


In [11]:
# your code here


**2.** Work out the affinity of columns declared `CHARACTER(10)`, `DOUBLE PRECISION`, `POINT` and
`BOOLEAN`, then check your answer by inserting the text `'1'` into each and printing its `typeof`.


In [12]:
# your code here


**3.** Create a `STRICT` table with a `TEXT` column and an `ANY` column, insert the integer 42 into
both, then the float 4.5 into both, and print every value with its `typeof`.


In [13]:
# your code here


**4.** Create a flexible table whose `celsius REAL` column has a `CHECK` that allows only REAL and
`NULL`, and show which of `-3.5`, `'4.5'` and `'n/a'` it accepts.


In [14]:
# your code here


**5.** Compress Oslo's readings for 1 March, written as CSV text, into a `BLOB` column, then read the
first two bytes back with `blobopen` and the whole value with `SELECT`, and print the first line of
the text.


In [15]:
# your code here


**6.** Print every table in the database with whether it is `STRICT`, from `pragma_table_list`,
leaving out SQLite's own tables, whose names begin with `sqlite_`.


In [16]:
# your code here


## Common errors

### sqlite3.IntegrityError: cannot store TEXT value in INTEGER column strict_readings.station_id


In [17]:
row = {"station": "Kirkenes", "hour": "2025-12-01T07:00", "celsius": "-8,8"}
conn.execute("INSERT INTO strict_readings (station_id, hour, celsius) VALUES (?, ?, ?)",
             (row["station"], row["hour"], parse_celsius(row["celsius"])))


IntegrityError: cannot store TEXT value in INTEGER column strict_readings.station_id

The export names its station, and `station_id` wants the station's id. The STRICT table could not
turn `'Kirkenes'` into an integer, so it refused the row, and the message names the table and column.
A flexible table would have stored `'Kirkenes'` in `station_id` without a word, and every join on
that column would have missed the row. Look the id up first:


In [18]:
conn.execute("INSERT INTO strict_readings (station_id, hour, celsius) VALUES (?, ?, ?)",
             (station_ids[row["station"]], row["hour"], parse_celsius(row["celsius"])))
conn.commit()

print(conn.execute("SELECT station_id, hour, celsius FROM strict_readings WHERE hour = ?", (row["hour"],)).fetchone())


(5, '2025-12-01T07:00', -8.8)


### sqlite3.OperationalError: unknown datatype for daily_summaries.day: "DATETIME"


In [19]:
conn.execute("""
    CREATE TABLE daily_summaries (
        station_id INTEGER NOT NULL,
        day        DATETIME NOT NULL,
        coldest    REAL
    ) STRICT
""")


OperationalError: unknown datatype for daily_summaries.day: "DATETIME"

A STRICT table allows only `INT`, `INTEGER`, `REAL`, `TEXT`, `BLOB` and `ANY`, and `DATETIME`, like
`VARCHAR(20)` or `BOOLEAN`, is none of them. SQLite has no date type: a date is stored as text, as
the readings' hours are, or as a number, which the **Adapters and Converters** notebook covers.
Declare the column as what it holds:


In [20]:
conn.execute("""
    CREATE TABLE daily_summaries (
        station_id INTEGER NOT NULL,
        day        TEXT NOT NULL,
        coldest    REAL
    ) STRICT
""")

print(conn.execute("SELECT strict FROM pragma_table_list WHERE name = 'daily_summaries'").fetchone())


(1,)


### TypeError: '>' not supported between instances of 'str' and 'float'


In [21]:
readings = [celsius for (celsius,) in conn.execute("SELECT celsius FROM real_column WHERE celsius IS NOT NULL")]
print(max(readings))


TypeError: '>' not supported between instances of 'str' and 'float'

`real_column` is declared `REAL`, but it kept `'n/a'` as text, and Python received that value as a
`str` among the floats. `max` compares its values in pairs, and Python refuses to compare text with a
number. The mistake happened when `'n/a'` was stored, long before this line. Find the values that are
not numbers by their storage class, and deal with them where they came in:


In [22]:
not_numbers = conn.execute("SELECT celsius, typeof(celsius) FROM real_column WHERE typeof(celsius) NOT IN ('real', 'null')")
print(not_numbers.fetchall())

numbers = [celsius for (celsius,) in conn.execute("SELECT celsius FROM real_column WHERE typeof(celsius) = 'real'")]
print(max(numbers))


[('n/a', 'text')]
10.0


### No error, and one freezing hour of three: readings in a column with no declared type


In [23]:
conn.execute("CREATE TABLE arrivals (station, hour, celsius)")
from_csv = [("Kirkenes", "2025-12-02T00:00", "-7.4"), ("Kirkenes", "2025-12-02T01:00", "-7.9"),
            ("Kirkenes", "2025-12-02T02:00", "0.3")]
from_api = [("Kirkenes", "2025-12-02T03:00", -8.3), ("Kirkenes", "2025-12-02T04:00", 1.2)]
conn.executemany("INSERT INTO arrivals VALUES (?, ?, ?)", from_csv + from_api)

print("hours below freezing:", conn.execute("SELECT COUNT(*) FROM arrivals WHERE celsius < 0").fetchone()[0])
print("stored as:", conn.execute("SELECT typeof(celsius), COUNT(*) FROM arrivals GROUP BY 1 ORDER BY 1").fetchall())


hours below freezing: 1
stored as: [('real', 2), ('text', 3)]


Three of the five readings were below freezing, and the query counted one. The table was created
with no types, so its columns have BLOB affinity and convert nothing: the readings from the CSV file
stayed text, text sorts after every number, and `< 0` is never true for it. Only the reading that
arrived from the API as a `float` was counted. Declare the column's type, and the same rows convert
on the way in:


In [24]:
conn.execute("CREATE TABLE typed_arrivals (station TEXT NOT NULL, hour TEXT NOT NULL, celsius REAL)")
conn.executemany("INSERT INTO typed_arrivals VALUES (?, ?, ?)", from_csv + from_api)

print("hours below freezing:", conn.execute("SELECT COUNT(*) FROM typed_arrivals WHERE celsius < 0").fetchone()[0])
print("stored as:", conn.execute("SELECT typeof(celsius), COUNT(*) FROM typed_arrivals GROUP BY 1 ORDER BY 1").fetchall())


hours below freezing: 3
stored as: [('real', 5)]


### No error, and 150 where the postal code was 0150: a column declared STRING


In [25]:
conn.execute("CREATE TABLE postcodes (town TEXT NOT NULL, code STRING NOT NULL)")
conn.executemany("INSERT INTO postcodes VALUES (?, ?)", [("Oslo", "0150"), ("Bergen", "5003")])

print(conn.execute("SELECT town, code, typeof(code) FROM postcodes").fetchall())
print("looked up by '0150':", conn.execute("SELECT town FROM postcodes WHERE code = ?", ("0150",)).fetchall())


[('Oslo', 150, 'integer'), ('Bergen', 5003, 'integer')]
looked up by '0150': [('Oslo',)]


`STRING` matches none of the affinity rules, so the column has NUMERIC affinity, and `'0150'` looked
like a number: it was stored as the integer 150, and its zero is gone. The lookup by `'0150'` still
worked, since the comparison converted that text the same way, which hides the damage until the code
leaves the database, on a label, in a file, or in a join with a table that stored it as text. A code
made of digits is text, so declare it `TEXT`, which keeps it as written:


In [26]:
conn.execute("CREATE TABLE text_postcodes (town TEXT NOT NULL, code TEXT NOT NULL)")
conn.executemany("INSERT INTO text_postcodes VALUES (?, ?)", [("Oslo", "0150"), ("Bergen", "5003")])

print(conn.execute("SELECT town, code, typeof(code) FROM text_postcodes").fetchall())


[('Oslo', '0150', 'text'), ('Bergen', '5003', 'text')]


### No error, and -7.0 where the reading was -7,4: CAST reads only the number at the start


In [27]:
conn.execute("CREATE TABLE imported (hour TEXT NOT NULL, celsius REAL)")
conn.executemany("INSERT INTO imported VALUES (?, ?)",
                 [("2025-12-03T00:00", "-7,4"), ("2025-12-03T01:00", "n/a"), ("2025-12-03T02:00", "-8.1")])

conn.execute("UPDATE imported SET celsius = CAST(celsius AS REAL) WHERE typeof(celsius) = 'text'")
print(conn.execute("SELECT hour, celsius, typeof(celsius) FROM imported ORDER BY hour").fetchall())


[('2025-12-03T00:00', -7.0, 'real'), ('2025-12-03T01:00', 0.0, 'real'), ('2025-12-03T02:00', -8.1, 'real')]


The `UPDATE` was meant to repair the text that a `REAL` column had kept, and every value now looks
like a number. But `CAST` reads the number at the start of the text and ignores the rest, so
`'-7,4'` became -7.0, losing its decimal, and `'n/a'`, which starts with no number at all, became
0.0, a reading of freezing point that nobody took. Nothing raised. Repair the rows in Python, where a
parser can say what the text means and refuse what it cannot read:


In [28]:
conn.execute("DELETE FROM imported")
conn.executemany("INSERT INTO imported VALUES (?, ?)",
                 [("2025-12-03T00:00", "-7,4"), ("2025-12-03T01:00", "n/a"), ("2025-12-03T02:00", "-8.1")])

text_rows = conn.execute("SELECT hour, celsius FROM imported WHERE typeof(celsius) = 'text'").fetchall()
conn.executemany("UPDATE imported SET celsius = ? WHERE hour = ?", [(parse_celsius(text), hour) for hour, text in text_rows])
conn.commit()

print(conn.execute("SELECT hour, celsius, typeof(celsius) FROM imported ORDER BY hour").fetchall())
conn.close()


[('2025-12-03T00:00', -7.4, 'real'), ('2025-12-03T01:00', None, 'null'), ('2025-12-03T02:00', -8.1, 'real')]


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [29]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A value's storage class, `NULL`, `INTEGER`, `REAL`, `TEXT` or `BLOB`, belongs to the value, and
  `typeof` reports it, whatever the column's declared type.
- A column's affinity comes from its declared type's name: `VARCHAR` is TEXT, `DOUBLE` is REAL, no
  type is BLOB, and `STRING`, `DATETIME` and `BOOLEAN` are NUMERIC.
- Affinity converts a value only when nothing is lost, so `'42'` becomes a number and `'n/a'` stays
  text in any flexible column.
- Numbers sort before text, so text among numbers sorts last, wins `MAX` and fails every `< 0`, and
  Python receives it as a `str`.
- A `STRICT` table allows six type names, converts what converts without loss, and raises
  `IntegrityError` for the rest, and opens only in SQLite 3.37.0 or later.
- A `BLOB` column holds bytes, and `blobopen` reads part of one without loading it all.
- Convert text in Python before it reaches the database, with a `STRICT` table to refuse whatever
  slips past, and never repair text with `CAST`.


## What is next

The **Adapters and Converters** notebook deals with values no storage class holds, such as a
`datetime` or a `Decimal`: how sqlite3 turns them into something SQLite can store, and back into
Python on the way out.


---

&#8592; **Previous:** [Row Factories](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/05-row-factories.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Adapters and Converters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/07-adapters-and-converters.ipynb) &#8594;
